# Experiment 12.0 — Local Primitive Bottleneck

Analysis-only notebook. Training and post-hoc extraction are performed by the Slurm workflow; this notebook reads finalized CSV artifacts.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_12_0_local_primitive_bottleneck' / 'local_primitive_bottleneck_v1'
dense = pd.read_csv(ART / 'dense_runs.csv')
dense_summary = pd.read_csv(ART / 'dense_summary.csv')
posthoc = pd.read_csv(ART / 'posthoc_runs.csv')
posthoc_summary = pd.read_csv(ART / 'posthoc_summary.csv')
dense_summary


## Dense T0 / RAW / EMA comparison


In [ ]:
dense_summary.sort_values(['method', 'temporal_mode'])[['temporal_mode','method','test_ba_mean','test_ba_std','l2_fixed250_probe_test_ba_mean','l2_wholecount_probe_test_ba_mean','primitive_effective_k_mean']]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for method, frame in dense.groupby('method'):
    means = frame.groupby('temporal_mode', sort=False)['test_ba'].mean().reindex(['t0','raw','ema'])
    ax.plot(means.index, means.values, marker='o', label=method)
ax.set_xlabel('Temporal evidence mode')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Dense representation performance')
ax.legend()
fig.tight_layout()


## Fixed-transform R0 → R2 diagnosis


In [ ]:
fixed_names = ['r0_direct_l2','r0a_integrated_128d','r0b_analog_16d','r1_main_what_posthoc','r2_main_activity_x_what_posthoc','r1_legacy_raw_softmax_posthoc','r2_legacy_rms_x_raw_softmax_posthoc']
fixed = posthoc[posthoc['representation'].isin(fixed_names)]
fixed.groupby(['temporal_mode','representation'], sort=False)['test_ba'].agg(['mean','std']).reset_index()


## R3/R4 sparsity–accuracy trade-off


In [ ]:
sparse = posthoc[posthoc['representation'].isin(['r3_gate','r4_peak'])].copy()
fig, ax = plt.subplots(figsize=(8, 5))
for (mode, rep), frame in sparse.groupby(['temporal_mode','representation']):
    summary = frame.groupby('quantile').agg(test_ba=('test_ba','mean'), events=('events_per_sample_test_mean','mean')).reset_index()
    ax.plot(summary['events'], summary['test_ba'], marker='o', label=f'{mode}/{rep}')
ax.set_xlabel('Events per sample')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Sparse primitive-event Pareto curve')
ax.legend()
fig.tight_layout()


## Token/shortcut diagnostics


In [ ]:
diagnostic_names = ['r2_dense','r2_hard_token','magnitude_only','r4_count_only']
posthoc[posthoc['representation'].isin(diagnostic_names)].groupby(['temporal_mode','source_method','representation'], sort=False)['test_ba'].agg(['mean','std']).reset_index()
